# Laboratório 2 — Extração de Características (*Features*) com SIFT

**Disciplina:** ESZA019 — Visão Computacional — UFABC 2026.2  
**Professor:** Celso Setsuo Kurashima

### Integrantes da equipe
- Lucas Rodrigues Teixeira — RA 11202131394
- Pedro Henrique Garcez Silva — RA 11202130642
- Roberto Sene Azevedo — RA 11202020360

**Data de realização dos experimentos:** 08/06/2026  
**Data de publicação do relatório:** 14/06/2026

## Sumário

1. [Introdução](#1-introdução)
2. [Fundamentação teórica](#2-fundamentação-teórica)
   - 2.1 [O que é uma *feature*?](#21-o-que-é-uma-feature)
   - 2.2 [Detector de cantos de Harris](#22-detector-de-cantos-de-harris)
   - 2.3 [Detector de Shi-Tomasi](#23-detector-de-shi-tomasi)
   - 2.4 [SIFT — Scale-Invariant Feature Transform](#24-sift--scale-invariant-feature-transform)
   - 2.5 [Correspondência (*matching*) e homografia](#25-correspondência-matching-e-homografia)
3. [Ambiente e instruções de reprodução](#3-ambiente-e-instruções-de-reprodução)
4. [Procedimentos experimentais (Parte 2)](#4-procedimentos-experimentais-parte-2)
   - 4.1 [(A) Feature Matching + Homography em duas imagens gravadas](#41-a-feature-matching--homography-em-duas-imagens-gravadas)
   - 4.2 [(B) Correspondência SIFT entre duas webcams](#42-b-correspondência-sift-entre-duas-webcams)
5. [Análise e discussão](#5-análise-e-discussão)
6. [Conclusões](#6-conclusões)
7. [Referências](#7-referências)

## 1. Introdução

No Laboratório 1 a equipe explorou o ciclo básico de **entrada, processamento e saída** de imagens e
vídeos com o OpenCV. Neste Laboratório 2 avançamos para um problema central da Visão Computacional: a
**extração de características** (*features*) e o seu uso para **reconhecer e localizar um mesmo objeto**
em imagens diferentes.

Uma *feature* é um ponto da imagem cuja vizinhança é suficientemente distinta para ser **detectada de
forma repetível** e **descrita de forma robusta**, mesmo quando a imagem sofre mudanças de escala,
rotação, iluminação ou ponto de vista. Cantos, quinas e regiões de textura rica são bons exemplos;
regiões planas (parede lisa) ou bordas retas são exemplos ruins, pois não permitem localizar o ponto
de forma única.

Este relatório está organizado em duas frentes, conforme o roteiro. A **Parte 1** apresenta a
fundamentação teórica dos principais detectores estudados (Harris, Shi-Tomasi e SIFT). A **Parte 2**
descreve os dois programas implementados pela equipe: (A) um programa que realiza *Feature Matching +
Homography to find Objects* sobre **duas imagens previamente gravadas** do mesmo objeto, e (B) uma
variação do mesmo código que executa a correspondência SIFT em tempo real lendo **duas webcams**.
Ao final, discutimos referências e aplicações da detecção de *features* no nosso trabalho final.

## 2. Fundamentação teórica

### 2.1 O que é uma *feature*?

Quando uma pessoa monta um quebra-cabeça, ela não compara cada peça pixel a pixel: ela procura **pontos
marcantes** — um canto, um encontro de cores, um padrão único — e tenta encontrá-los em outras peças.
A Visão Computacional segue a mesma intuição. Chamamos esses pontos marcantes de **features**, **pontos
de interesse** ou **keypoints**.

O que torna um ponto uma boa *feature*? A ideia central é a **variação local da intensidade** em torno
do ponto, em diferentes direções:

| Tipo de região | Comportamento ao deslocar uma janela | É boa feature? |
|----------------|--------------------------------------|----------------|
| **Região plana** (textura uniforme) | a intensidade quase não muda em nenhuma direção | Não — impossível localizar |
| **Borda** (aresta) | muda muito ao cruzar a borda, mas pouco ao deslizar ao longo dela | Parcialmente — ambígua ao longo da borda |
| **Canto** | muda significativamente em **todas** as direções | Sim — localização única |

O processo se divide em duas etapas complementares: **detecção** (*feature detection*) — encontrar onde
estão os pontos de interesse — e **descrição** (*feature description*) — resumir a vizinhança de cada
ponto em um vetor numérico (o **descritor**) que permita compará-lo com pontos de outra imagem.

### 2.2 Detector de cantos de Harris

O **detector de Harris** (Chris Harris e Mike Stephens, 1988) formaliza matematicamente a noção de
"canto". Ele considera o deslocamento de uma pequena janela $(u, v)$ e mede a variação de intensidade
resultante:

$$E(u,v) = \sum_{x,y} w(x,y)\,[\,I(x+u, y+v) - I(x,y)\,]^2$$

onde $w(x,y)$ é uma janela de ponderação (em geral gaussiana). Expandindo por Taylor, a expressão se
aproxima de uma forma quadrática que depende da **matriz de estrutura** $M$, construída a partir das
derivadas $I_x$ e $I_y$ da imagem:

$$M = \sum_{x,y} w(x,y) \begin{bmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{bmatrix}$$

A partir dos autovalores $\lambda_1$ e $\lambda_2$ de $M$ define-se a **resposta de canto**:

$$R = \det(M) - k\,(\operatorname{tr} M)^2 = \lambda_1\lambda_2 - k\,(\lambda_1 + \lambda_2)^2$$

com $k$ tipicamente entre 0,04 e 0,06. A interpretação geométrica é direta: quando **ambos** os
autovalores são grandes ($R$ grande e positivo) há um **canto**; quando apenas um é grande ($R$
negativo) há uma **borda**; quando ambos são pequenos ($|R|$ pequeno) a região é **plana**. No OpenCV
isso é calculado por `cv.cornerHarris`. Uma propriedade importante é que o Harris é **invariante à
rotação**, mas **não é invariante à escala** — um canto detectado em uma imagem pequena pode parecer
uma curva suave (sem cantos) quando a imagem é muito ampliada.

### 2.3 Detector de Shi-Tomasi

Jianbo Shi e Carlo Tomasi (1994), no artigo *"Good Features to Track"*, propuseram uma pequena, porém
eficaz, modificação no critério de Harris. Em vez da combinação $\det(M) - k\,(\operatorname{tr} M)^2$,
eles definiram a resposta simplesmente como o **menor** dos dois autovalores:

$$R = \min(\lambda_1, \lambda_2)$$

A justificativa é intuitiva: se até o **menor** autovalor já é grande, então a janela varia bastante em
todas as direções e o ponto é, com certeza, um bom canto. Esse critério mostrou-se mais estável para
**rastreamento** de pontos entre quadros de vídeo. No OpenCV o método está disponível em
`cv.goodFeaturesToTrack`, que retorna os **N cantos mais fortes** da imagem — muito usado como ponto de
partida para algoritmos de fluxo óptico (Lucas-Kanade). Assim como o Harris, é invariante à rotação,
mas não à escala.

### 2.4 SIFT — Scale-Invariant Feature Transform

Os detectores anteriores falham quando a escala muda. O **SIFT**, proposto por David Lowe (1999/2004),
resolve exatamente esse problema, sendo **invariante a escala e rotação** e robusto a variações de
iluminação e a pequenas mudanças de ponto de vista. É o algoritmo central deste laboratório. Ele opera
em quatro grandes etapas:

1. **Detecção de extremos no espaço de escala.** A imagem é repetidamente suavizada com filtros
   gaussianos de $\sigma$ crescente, formando uma *pirâmide* organizada em **oitavas**. A diferença
   entre gaussianas vizinhas gera as imagens **DoG** (*Difference of Gaussians*), uma aproximação
   eficiente do Laplaciano do Gaussiano. Candidatos a keypoint são os pixels que são **máximos ou
   mínimos locais** comparados aos seus 8 vizinhos na mesma escala e aos 9+9 vizinhos nas escalas
   adjacente — ou seja, pontos que se destacam **em escala e espaço**.
2. **Refinamento e seleção dos keypoints.** Os candidatos são refinados para precisão subpixel
   (expansão de Taylor da DoG) e descartam-se os de **baixo contraste** e os situados sobre **bordas**
   (usando uma razão de curvaturas principais, análoga ao Harris), mantendo apenas pontos estáveis.
3. **Atribuição de orientação.** A cada keypoint é atribuída uma ou mais orientações dominantes,
   calculadas a partir de um **histograma de gradientes** da vizinhança. Descrever o ponto **relativo
   a essa orientação** é o que confere ao SIFT a **invariância à rotação**.
4. **Descritor do keypoint.** Em torno do ponto, e na sua escala/orientação, monta-se uma grade de
   $4\times4$ subregiões; em cada uma calcula-se um histograma de gradientes com 8 direções,
   produzindo um **vetor descritor de 128 dimensões** ($4\times4\times8$), normalizado para reduzir
   o efeito de mudanças de iluminação.

No OpenCV o detector é criado com `cv.SIFT_create()` e os keypoints e descritores são obtidos com
`sift.detectAndCompute(img, None)`. Observação histórica útil: o SIFT já foi patenteado e ficava no
módulo *contrib* (`xfeatures2d`); com a expiração da patente, nas versões recentes do OpenCV 4.x ele
passou a integrar o módulo principal, exatamente como usamos aqui.

### 2.5 Correspondência (*matching*) e homografia

Detectar e descrever *features* é só metade do trabalho. Para **localizar um objeto** em uma cena
precisamos **casar** os descritores de uma imagem (a *query*, que contém o objeto isolado) com os da
outra (a *train*, que contém a cena). Duas peças são fundamentais:

- **FLANN** (*Fast Library for Approximate Nearest Neighbors*): para cada descritor da *query*, busca
  rapidamente os vizinhos mais próximos entre os descritores da cena. Usamos `knnMatch(..., k=2)`, que
  retorna os **dois** vizinhos mais próximos de cada ponto.
- **Teste de razão de Lowe** (*ratio test*): um *match* só é considerado bom se o melhor vizinho for
  claramente mais próximo que o segundo, isto é, `m.distance < 0.7 * n.distance`. Isso elimina a maior
  parte das correspondências ambíguas/erradas.

Com um conjunto de bons *matches*, estimamos a **homografia** — a transformação projetiva $3\times3$ que
mapeia o plano do objeto na *query* para o plano dele na cena:

$$\begin{bmatrix} x' \\ y' \\ 1 \end{bmatrix} \sim H \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}$$

A homografia é calculada com `cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)`. O **RANSAC** é
essencial: ele estima $H$ usando apenas os *inliers* (correspondências geometricamente coerentes) e
ignora os *outliers* restantes. Por fim, projetamos os quatro cantos da *query* pela homografia com
`cv.perspectiveTransform` e desenhamos esse contorno sobre a cena — é a moldura que **localiza o
objeto** no resultado.

## 3. Ambiente e instruções de reprodução

Os experimentos foram executados em **Linux**, com **Python 3** e **OpenCV 4.x**. O SIFT está incluído
no pacote `opencv-python` recente (não é mais necessário o `opencv-contrib-python`). Para reproduzir:

```bash
# 1) Criar/ativar o ambiente virtual
python3 -m venv venv
source venv/bin/activate

# 2) Instalar as dependências
pip install opencv-python numpy matplotlib

# 3) Executar os programas da Parte 2
python3 feature_matching.py        # item (A) — duas imagens gravadas
python3 feature_matching_cam.py    # item (B) — duas webcams
```

> **Observação:** as funções de captura de webcam e as janelas (`cv.imshow`) exigem uma máquina com
> **interface gráfica** e, no item (B), **duas câmeras** disponíveis (índices `0` e `1`). As células de
> código abaixo reproduzem fielmente os programas entregues pela equipe; os arquivos de entrada
> (`box.jpg`, `box_in_scene.jpeg`) e as imagens de resultado estão na mesma pasta deste notebook e são
> referenciados por **caminhos relativos**.

## 4. Procedimentos experimentais (Parte 2)

### 4.1 (A) Feature Matching + Homography em duas imagens gravadas

**Objetivo:** implementar o programa do tutorial *Feature Matching + Homography to find Objects* do
OpenCV, lendo **duas imagens previamente gravadas** que contêm o mesmo objeto em posições/condições
distintas, e exibir as correspondências obtidas com a moldura do objeto localizado na cena.

**Objeto escolhido pela equipe:** uma caixa do cereal **NESCAU** (Nestlé). A imagem *query* (`box.jpg`)
é uma foto frontal e limpa da embalagem; a imagem da cena (`box_in_scene.jpeg`) é uma foto promocional
em que a mesma embalagem aparece **menor, rotacionada e em outro fundo** (uma mão segurando o produto
em um corredor de mercado). Essa diferença de escala e de fundo é justamente o cenário em que o SIFT
se destaca.

**Fluxo do programa:** (1) ler as duas imagens em tons de cinza; (2) detectar keypoints e descritores
SIFT em ambas; (3) casar os descritores com FLANN (`knnMatch`, k=2); (4) filtrar pelos *good matches*
com o teste de razão de Lowe (0,7); (5) se houver pelo menos `MIN_MATCH_COUNT = 10` boas
correspondências, estimar a **homografia** com RANSAC e projetar os cantos da *query* para desenhar a
moldura na cena; (6) exibir as correspondências com `cv.drawMatches`.

In [ ]:
import numpy as np
import cv2 as cv
from matplotlib import pyplot as plt

MIN_MATCH_COUNT = 10

img1 = cv.imread('box.jpg', cv.IMREAD_GRAYSCALE)          # queryImage (objeto isolado)
img2 = cv.imread('box_in_scene.jpeg', cv.IMREAD_GRAYSCALE) # trainImage (cena)

# Inicia o detector SIFT
sift = cv.SIFT_create()

# Encontra keypoints e descritores com SIFT nas duas imagens
kp1, des1 = sift.detectAndCompute(img1, None)
kp2, des2 = sift.detectAndCompute(img2, None)

# Configuracao do matcher FLANN (KD-Tree para descritores SIFT)
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)

flann = cv.FlannBasedMatcher(index_params, search_params)

matches = flann.knnMatch(des1, des2, k=2)

# Guarda os bons matches segundo o teste de razao de Lowe
good = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        good.append(m)

if len(good) > MIN_MATCH_COUNT:
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

    # Estima a homografia robusta com RANSAC
    M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)
    matchesMask = mask.ravel().tolist()

    # Projeta os 4 cantos da query para desenhar a moldura do objeto na cena
    h, w = img1.shape
    pts = np.float32([[0, 0], [0, h - 1], [w - 1, h - 1], [w - 1, 0]]).reshape(-1, 1, 2)
    dst = cv.perspectiveTransform(pts, M)

    img2 = cv.polylines(img2, [np.int32(dst)], True, 255, 3, cv.LINE_AA)

else:
    print("Not enough matches are found - {}/{}".format(len(good), MIN_MATCH_COUNT))
    matchesMask = None

draw_params = dict(matchColor=(0, 255, 0),   # desenha os matches em verde
                   singlePointColor=None,
                   matchesMask=matchesMask,   # desenha apenas os inliers
                   flags=2)

img3 = cv.drawMatches(img1, kp1, img2, kp2, good, None, **draw_params)

plt.imshow(img3, 'gray'), plt.show()

**Resultado obtido pela equipe.** A figura abaixo mostra a saída do programa. À esquerda está a *query*
(a caixa do Nescau isolada) e à direita a cena; as **linhas verdes** ligam os keypoints
correspondentes (apenas os *inliers* aprovados pelo RANSAC), e a **moldura branca** sobre a cena marca
a região onde o objeto foi localizado pela homografia. Mesmo com a embalagem aparecendo em escala
menor, parcialmente rotacionada e sobre um fundo completamente diferente, o SIFT encontrou
correspondências suficientes para localizar o produto corretamente.

![Correspondências SIFT + homografia entre as duas imagens da caixa de Nescau](caixa_matching.png)

*Figura 1 — Saída de `feature_matching.py`: correspondências (verde) e objeto localizado (moldura branca).*

### 4.2 (B) Correspondência SIFT entre duas webcams

**Objetivo:** modificar o programa do item (A) para que, em vez de ler dois arquivos, ele leia
**duas webcams pelo mesmo código** e execute a correspondência SIFT **a cada par de quadros**,
mostrando o mesmo tipo de resultado em uma janela de vídeo contínua.

**Principais alterações em relação ao item (A):**
- As duas fontes passam a ser `cv.VideoCapture(0)` e `cv.VideoCapture(1)` (webcam interna e externa),
  com verificação de abertura de cada câmera.
- Todo o processamento (cinza → SIFT → FLANN → ratio test → homografia → desenho) é colocado dentro de
  um **laço `while`** que processa os quadros em tempo real.
- Foi adicionada **robustez** para o cenário ao vivo: o código verifica se `des1`/`des2` existem e têm
  pelo menos 2 descritores antes de chamar o `knnMatch`, e trata o caso em que `knnMatch` devolve menos
  de 2 vizinhos — situações comuns quando uma câmera aponta para uma superfície sem textura. Sem essas
  verificações, o programa quebraria com exceção quando não houvesse keypoints suficientes.
- A saída é exibida continuamente com `cv.imshow` em uma única janela (`SIFT - Duas Webcams`); a tecla
  **`q`** encerra o programa e libera as duas câmeras.

In [ ]:
import numpy as np
import cv2 as cv

MIN_MATCH_COUNT = 10

# Abre duas webcams
# Normalmente:
# 0 = webcam principal/notebook
# 1 = webcam externa
cap1 = cv.VideoCapture(0)
cap2 = cv.VideoCapture(1)

if not cap1.isOpened():
    print("Erro: nao foi possivel abrir a webcam 0")
    exit()

if not cap2.isOpened():
    print("Erro: nao foi possivel abrir a webcam 1")
    exit()

# Cria detector SIFT
sift = cv.SIFT_create()

# Configuracao do FLANN
FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)

flann = cv.FlannBasedMatcher(index_params, search_params)

while True:
    # Le frames das duas webcams
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()

    if not ret1 or not ret2:
        print("Erro ao capturar frame de uma das webcams")
        break

    # Converte para escala de cinza
    img1 = cv.cvtColor(frame1, cv.COLOR_BGR2GRAY)
    img2 = cv.cvtColor(frame2, cv.COLOR_BGR2GRAY)

    # Detecta keypoints e descritores
    kp1, des1 = sift.detectAndCompute(img1, None)
    kp2, des2 = sift.detectAndCompute(img2, None)

    matchesMask = None
    good = []

    # Evita erro caso uma das imagens nao tenha descritores suficientes
    if des1 is not None and des2 is not None and len(des1) >= 2 and len(des2) >= 2:

        matches = flann.knnMatch(des1, des2, k=2)

        # Lowe's ratio test
        for match in matches:
            if len(match) == 2:
                m, n = match
                if m.distance < 0.7 * n.distance:
                    good.append(m)

        if len(good) > MIN_MATCH_COUNT:
            src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
            dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

            M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)

            if M is not None and mask is not None:
                matchesMask = mask.ravel().tolist()

                h, w = img1.shape
                pts = np.float32([[0, 0], [0, h - 1], [w - 1, h - 1], [w - 1, 0]]).reshape(-1, 1, 2)
                dst = cv.perspectiveTransform(pts, M)

                # Desenha o contorno da regiao correspondente na imagem 2
                img2 = cv.polylines(img2, [np.int32(dst)], True, 255, 3, cv.LINE_AA)
        else:
            print(f"Not enough matches are found - {len(good)}/{MIN_MATCH_COUNT}")
    else:
        print("Descritores insuficientes em uma das webcams")

    draw_params = dict(matchColor=(0, 255, 0),
                       singlePointColor=None,
                       matchesMask=matchesMask,
                       flags=2)

    # Desenha correspondencias entre os dois frames
    img_matches = cv.drawMatches(img1, kp1, img2, kp2, good, None, **draw_params)

    # Mostra resultado em janela de video
    cv.imshow("SIFT - Duas Webcams", img_matches)

    # Aperte q para sair
    if cv.waitKey(1) & 0xFF == ord('q'):
        break

# Libera recursos
cap1.release()
cap2.release()
cv.destroyAllWindows()

**Resultado obtido pela equipe.** A captura de tela abaixo mostra a janela de vídeo em funcionamento. À
esquerda está o quadro da webcam 0 (integrantes da equipe) e à direita o quadro da webcam 1, que apontou
para um **celular exibindo a propaganda do Nescau** — o mesmo objeto do item (A). As linhas verdes ligam
os keypoints correspondentes entre os dois quadros em tempo real, e a moldura marca a região do objeto
detectada pela homografia. A correspondência é recalculada a cada par de quadros lidos das câmeras.

![Janela de vídeo da correspondência SIFT entre duas webcams](SIFT%20-%20Duas%20Webcams_screenshot_08.06.2026.png)

*Figura 2 — Captura de `feature_matching_cam.py` (08/06/2026): correspondência SIFT ao vivo entre duas webcams.*

## 5. Análise e discussão

**Funcionamento e comparação dos detectores.** Os experimentos confirmaram, na prática, a vantagem do
SIFT sobre detectores como Harris e Shi-Tomasi para a tarefa de localizar objetos: enquanto Harris e
Shi-Tomasi são invariantes apenas à rotação, o SIFT é também **invariante à escala**, o que foi
decisivo no item (A), em que a caixa de Nescau aparecia em **tamanhos bem diferentes** nas duas
imagens. O *ratio test* de Lowe (limiar 0,7) e o **RANSAC** na estimativa da homografia foram os
responsáveis por descartar as correspondências erradas e manter apenas os *inliers* geometricamente
coerentes — sem eles, a moldura do objeto saltaria ou apareceria distorcida.

**Dificuldades encontradas e soluções.**
- *Migração do SIFT:* em versões antigas o `cv.SIFT_create()` ficava em `cv.xfeatures2d` e exigia
  `opencv-contrib-python`. Confirmamos que, no OpenCV 4.x atual, o SIFT está no módulo principal,
  bastando `pip install opencv-python`.
- *Robustez no item (B):* ao apontar uma câmera para uma parede lisa, `detectAndCompute` retornava
  poucos (ou nenhum) descritores e o `knnMatch` quebrava. A solução foi **verificar** `des1`/`des2` e o
  tamanho de cada `match` antes de processar, evitando exceções e mantendo o vídeo fluido.
- *Índices das câmeras:* a webcam interna nem sempre é o índice `0`; foi preciso testar `0`/`1` para
  identificar corretamente cada dispositivo. O código já trata o caso de uma câmera não abrir.
- *Iluminação e textura:* superfícies sem textura geram poucos keypoints. Objetos com padrões ricos
  (como a embalagem colorida do Nescau) produzem correspondências muito mais estáveis.

**Aplicações da detecção de *features* e uso no trabalho final.** A literatura e os projetos abertos
mostram que detecção e correspondência de *features* sustentam aplicações como: **costura de imagens**
(*panorama stitching*), **realidade aumentada** (fixar conteúdo virtual sobre um marcador real),
**odometria visual e SLAM** em robótica e veículos autônomos, **reconhecimento e rastreamento de
objetos**, e **registro de imagens** médicas e de sensoriamento remoto. Para o **trabalho final**, a
equipe pretende reaproveitar exatamente o pipeline deste laboratório — SIFT + FLANN + ratio test +
homografia via RANSAC — como base para **reconhecer e rastrear um objeto-alvo** capturado pela câmera,
podendo evoluir para alternativas mais rápidas quando o desempenho em tempo real for crítico
(por exemplo **ORB**, livre de patente e mais leve, ou descritores aprendidos por redes neurais).

## 6. Conclusões

O Laboratório 2 cumpriu seus objetivos. A equipe estudou os fundamentos da **extração de**
**características** — o conceito de *feature*, os detectores de **Harris** e **Shi-Tomasi** e,
principalmente, o **SIFT** — e os aplicou em dois programas OpenCV. No item (A), o programa de
*Feature Matching + Homography* localizou corretamente uma caixa de Nescau em uma cena com escala,
rotação e fundo diferentes, evidenciando a invariância do SIFT. No item (B), o mesmo pipeline foi
adaptado para ler **duas webcams** e executar a correspondência **em tempo real**, exibindo o resultado
quadro a quadro em uma janela de vídeo. Ficou claro o papel de cada componente: o SIFT garante
keypoints/descritores robustos; o FLANN com o *ratio test* de Lowe seleciona boas correspondências; e a
homografia com RANSAC localiza o objeto de forma geometricamente coerente. Esse conjunto de técnicas
forma uma base sólida e diretamente reaproveitável para o trabalho final da disciplina.

## 7. Referências

1. OpenCV — *Feature Detection and Description (índice)*. Disponível em:
   https://docs.opencv.org/4.x/db/d27/tutorial_py_table_of_contents_feature2d.html
2. OpenCV — *Understanding Features*. Disponível em:
   https://docs.opencv.org/4.x/df/d54/tutorial_py_features_meaning.html
3. OpenCV — *Harris Corner Detection*. Disponível em:
   https://docs.opencv.org/4.x/dc/d0d/tutorial_py_features_harris.html
4. OpenCV — *Shi-Tomasi Corner Detector & Good Features to Track*. Disponível em:
   https://docs.opencv.org/4.x/d4/d8c/tutorial_py_shi_tomasi.html
5. OpenCV — *Introduction to SIFT (Scale-Invariant Feature Transform)*. Disponível em:
   https://docs.opencv.org/4.x/da/df5/tutorial_py_sift_intro.html
6. OpenCV — *Feature Matching + Homography to find Objects*. Disponível em:
   https://docs.opencv.org/4.x/d1/de0/tutorial_py_feature_homography.html
7. LOWE, D. G. *Distinctive Image Features from Scale-Invariant Keypoints*. International Journal of
   Computer Vision, v. 60, n. 2, p. 91–110, 2004.
8. HARRIS, C.; STEPHENS, M. *A Combined Corner and Edge Detector*. Proceedings of the 4th Alvey Vision
   Conference, p. 147–151, 1988.
9. SHI, J.; TOMASI, C. *Good Features to Track*. IEEE Conference on Computer Vision and Pattern
   Recognition (CVPR), p. 593–600, 1994.